In [0]:

# ==========================================
# NOTEBOOK 2: SILVER DATA PROCESSING
# Retail Sales Data Engineering Pipeline
# ==========================================

# 1. IMPORT LIBRARIES

from pyspark.sql.functions import (
    col, trim, initcap, when, year, month,
    date_format, round as spark_round
)

from pyspark.sql.types import DecimalType


# 2. READ BRONZE DELTA TABLES

customer_bronze = spark.table(
    "retail_practice.default.bronze_customers"
)

product_bronze = spark.table(
    "retail_practice.default.bronze_products"
)

order_bronze = spark.table(
    "retail_practice.default.bronze_orders"
)

print("Bronze tables loaded successfully")


# 3. CLEAN CUSTOMER DATA

silver_customers = customer_bronze.dropDuplicates(
    ["customer_id"]
)

silver_customers = silver_customers.fillna({
    "email": "Unknown",
    "phone": "Unknown",
    "customer_segment": "Unknown"
}).withColumn(
    "city",
    initcap(trim(col("city")))
)


# 4. CLEAN PRODUCT DATA

silver_products = product_bronze.dropDuplicates(
    ["product_id"]
)

silver_products = silver_products.withColumn(
    "category",
    initcap(trim(col("category")))
)


# 5. CLEAN ORDER DATA

silver_orders = order_bronze.dropDuplicates(
    ["order_line_id"]
)

silver_orders = silver_orders.fillna({
    "discount_percent": 0,
    "payment_method": "Unknown"
})

print("Data cleaning completed")


# 6. CALCULATE SALES METRICS

silver_orders = silver_orders.withColumn(
    "Gross_sale",
    spark_round(
        col("quantity") * col("unit_price"), 2
    )
).withColumn(
    "discount_amount",
    spark_round(
        col("Gross_sale") * col("discount_percent") / 100, 2
    )
).withColumn(
    "Net_sale",
    spark_round(
        col("Gross_sale") - col("discount_amount"), 2
    )
)


# 7. CLASSIFY ORDER STATUS

silver_orders = silver_orders.withColumn(
    "sales_category",
    when(col("order_status") == "Delivered", "Completed")
    .when(col("order_status") == "Returned", "Returned")
    .when(col("order_status") == "Cancelled", "Cancelled")
    .otherwise("In Progress")
)


# 8. EXTRACT DATE ATTRIBUTES

silver_orders = silver_orders.withColumn(
    "order_year", year(col("order_date"))
).withColumn(
    "order_month", month(col("order_date"))
).withColumn(
    "Month_name", date_format(col("order_date"), "MMMM")
)

print("Business transformations completed")


# 9. INTEGRATE ORDERS WITH PRODUCTS AND CUSTOMERS

silver_sales = silver_orders.join(
    silver_products,
    on="product_id",
    how="inner"
).join(
    silver_customers,
    on="customer_id",
    how="inner"
)


# 10. SELECT FINAL SILVER COLUMNS

silver_retail_sales = silver_sales.select(
    "order_line_id",
    "order_id",
    "order_date",
    "customer_id",
    "first_name",
    "last_name",
    "city",
    "customer_segment",
    "product_id",
    "product_name",
    "category",
    "brand",
    "quantity",
    "unit_price",
    "discount_percent",
    "Gross_sale",
    "discount_amount",
    "Net_sale",
    "order_status",
    "sales_category",
    "payment_method",
    "shipping_cost",
    "delivery_date",
    "order_year",
    "order_month",
    "Month_name"
)

print("Data integration completed")


# 11. SAVE SILVER DELTA TABLE

silver_retail_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "retail_practice.default.silver_retail_sales"
    )

print("Silver Delta table saved successfully")


# 12. VALIDATE SAVED TABLE

saved_silver = spark.table(
    "retail_practice.default.silver_retail_sales"
)

print("----------------------------------")
print("SILVER PIPELINE VALIDATION")
print("----------------------------------")

print("Total rows:", saved_silver.count())
print("Total columns:", len(saved_silver.columns))

print(
    "Unique order lines:",
    saved_silver.select("order_line_id").distinct().count()
)

print("----------------------------------")

display(saved_silver.limit(10))
